In [1]:
from rdflib import Graph, RDF, RDFS, OWL, URIRef, BNode, Literal
from collections import Counter, defaultdict

In [2]:
file = "hi_ontology_linked.owl" 

In [4]:
SCENARIO_CLASS_KEYWORDS = {"scenario"}
SCENARIO_INSTANCE_KEYWORDS = {"scenario"}

In [5]:
# Main HI extension classes
KEY_HI_CLASSES = {
    "TrustLevel",
    "HighTrust", "MediumTrust", "LowTrust",
    "AutonomyLevel",
    "LowAutonomy", "SemiAutonomous", "HighAutonomy",
    "ControlMode",
    "HumanInTheLoop", "HumanOnTheLoop", "HumanOutOfTheLoop",
    "DecisionRole",
    "HumanFinalDecisionMaker", "AIFinalDecisionMaker", "SharedDecisionMaking",
    "ExplanationType",
    "OutcomeExplanation", "ProcessExplanation", "CounterfactualExplanation",
    "CoordinationPattern",
    "SequentialCoordination", "ParallelCoordination", "SupervisoryCoordination",
    "CollaborativeCoordination"
}

In [6]:
EXTERNAL_LINK_PREDICATES = {
    OWL.sameAs,
    RDFS.seeAlso
}

In [7]:
STANDARD_NAMESPACES = [
    str(RDF),
    str(RDFS),
    str(OWL),
    "http://www.w3.org/2001/XMLSchema#"
]

In [8]:
# LOAD GRAPH

g = Graph()
g.parse(file)


<Graph identifier=N4a9c01170eb64d7c85090a9cd70e6615 (<class 'rdflib.graph.Graph'>)>

In [9]:
def local_name(node):
    if isinstance(node, BNode):
        return str(node)
    s = str(node)
    if "#" in s:
        return s.split("#")[-1]
    return s.rstrip("/").split("/")[-1]

In [10]:
def get_namespace(uri):
    s = str(uri)
    if "#" in s:
        return s.rsplit("#", 1)[0] + "#"
    return s.rsplit("/", 1)[0] + "/"

In [11]:
def is_standard_namespace(uri):
    s = str(uri)
    return any(s.startswith(ns) for ns in STANDARD_NAMESPACES)

In [12]:
def is_schema_resource(node, graph):
    return (
        (node, RDF.type, OWL.Class) in graph or
        (node, RDF.type, RDFS.Class) in graph or
        (node, RDF.type, OWL.ObjectProperty) in graph or
        (node, RDF.type, OWL.DatatypeProperty) in graph or
        (node, RDF.type, OWL.AnnotationProperty) in graph or
        (node, RDF.type, RDF.Property) in graph or
        (node, RDF.type, OWL.Restriction) in graph or
        (node, RDF.type, OWL.Ontology) in graph
    )

In [13]:
def is_external_uri(uri):
    if not isinstance(uri, URIRef):
        return False
    s = str(uri)
    if s.startswith("file://"):
        return False
    return not is_standard_namespace(uri)

In [14]:
def matches_keyword_set(name, keywords):
    lname = name.lower()
    return any(k in lname for k in keywords)

# BASIC COUNTS

In [15]:
total_triples = len(g)

In [16]:
classes = set(g.subjects(RDF.type, OWL.Class)) | set(g.subjects(RDF.type, RDFS.Class))
object_properties = set(g.subjects(RDF.type, OWL.ObjectProperty))
datatype_properties = set(g.subjects(RDF.type, OWL.DatatypeProperty))
annotation_properties = set(g.subjects(RDF.type, OWL.AnnotationProperty))
generic_properties = set(g.subjects(RDF.type, RDF.Property))
restrictions = set(g.subjects(RDF.type, OWL.Restriction))

In [17]:
excluded_types = {
    OWL.Class,
    RDFS.Class,
    OWL.ObjectProperty,
    OWL.DatatypeProperty,
    OWL.AnnotationProperty,
    RDF.Property,
    OWL.Restriction,
    OWL.Ontology,
    OWL.AllDisjointClasses,
    OWL.AllDisjointProperties
}


In [18]:
individuals = set()
instance_types = defaultdict(set)

for s, _, o in g.triples((None, RDF.type, None)):
    if isinstance(s, URIRef) and o not in excluded_types and not is_schema_resource(s, g):
        individuals.add(s)
        instance_types[s].add(o)


In [ ]:
# class distribution
class_counter = Counter()
for inst, types in instance_types.items():
    for t in types:
        if t not in excluded_types:
            class_counter[t] += 1

In [20]:
scenario_instances = set()
for inst, types in instance_types.items():
    type_local_names = {local_name(t) for t in types}
    inst_local_name = local_name(inst)

    # Scenario detection
    if (
        any(matches_keyword_set(tn, SCENARIO_CLASS_KEYWORDS) for tn in type_local_names)
        or matches_keyword_set(inst_local_name, SCENARIO_INSTANCE_KEYWORDS)
    ):
        scenario_instances.add(inst)


In [21]:
external_links = []
external_target_uris = set()
external_link_sources = set()

for s, p, o in g:
    if p in EXTERNAL_LINK_PREDICATES and isinstance(o, URIRef) and is_external_uri(o):
        external_links.append((s, p, o))
        external_target_uris.add(o)
        external_link_sources.add(s)

external_link_pred_counts = Counter(local_name(p) for _, p, _ in external_links)


In [22]:
predicate_counter = Counter()
for _, p, _ in g:
    predicate_counter[p] += 1

In [ ]:
namespace_counter = Counter()
for s, p, o in g:
    for node in (s, p, o):
        if isinstance(node, URIRef):
            namespace_counter[get_namespace(node)] += 1

In [24]:
key_hi_counter = Counter()

for inst, types in instance_types.items():
    for t in types:
        lname = local_name(t)
        if lname in KEY_HI_CLASSES:
            key_hi_counter[lname] += 1

In [ ]:
# metrics
out_degree_counter = Counter()
in_degree_counter = Counter()

for s, p, o in g:
    if s in individuals:
        out_degree_counter[s] += 1
    if o in individuals:
        in_degree_counter[o] += 1

avg_out_degree = (
    sum(out_degree_counter.values()) / len(individuals)
    if individuals else 0
)

avg_in_degree = (
    sum(in_degree_counter.values()) / len(individuals)
    if individuals else 0
)

linked_individual_ratio = (
    len(external_link_sources) / len(individuals)
    if individuals else 0
)

In [30]:
print("\n[BASIC GRAPH SIZE]")
print(f"Total triples: {total_triples}")

print("\n[SCHEMA]")
print(f"Classes: {len(classes)}")
print(f"Object properties: {len(object_properties)}")
print(f"Datatype properties: {len(datatype_properties)}")
print(f"OWL restrictions: {len(restrictions)}")

print("\n[INSTANCE LAYER]")
print(f"Individuals: {len(individuals)}")
print(f"Detected scenario instances: {len(scenario_instances)}")
print(f"Average outgoing triples per individual: {avg_out_degree:.2f}")
print(f"Average incoming triples per individual: {avg_in_degree:.2f}")

print("\n[LINKING]")
print(f"External links (owl:sameAs / rdfs:seeAlso): {len(external_links)}")
print(f"Unique external target URIs: {len(external_target_uris)}")
print(f"Individuals with at least one external link: {len(external_link_sources)}")
print(f"Ratio of linked individuals: {linked_individual_ratio:.2%}")

print("\n[EXTERNAL LINK PREDICATES]")
for pred, count in external_link_pred_counts.most_common():
    print(f"{pred:30s} {count}")


print("\n[SAMPLE SCENARIO INSTANCES]")
for s in sorted(scenario_instances, key=lambda x: local_name(x))[:15]:
    print(f"- {local_name(s)}")




[BASIC GRAPH SIZE]
Total triples: 1142

[SCHEMA]
Classes: 47
Object properties: 27
Datatype properties: 0
OWL restrictions: 7

[INSTANCE LAYER]
Individuals: 219
Detected scenario instances: 13
Average outgoing triples per individual: 3.86
Average incoming triples per individual: 1.46

[LINKING]
External links (owl:sameAs / rdfs:seeAlso): 133
Unique external target URIs: 96
Individuals with at least one external link: 80
Ratio of linked individuals: 36.53%

[EXTERNAL LINK PREDICATES]
seeAlso                        101
sameAs                         32

[SAMPLE SCENARIO INSTANCES]
- AutonomousVehicleBeliefCommunication
- BRIDGET_HybridDecisionScenario
- EndUserInteractionWithAI
- FaceMatchingDecisionSupport
- HILO_LegalDesk_Scenario
- HI_DiagnosisAdjudicationScenario
- HI_EmergencyAntibioticsScenario
- HybridMoralDecisionMaking
- SHARPIE_HumanAI_RL_Scenario
- WarehouseAVGscenario
- clinical_scenario_1
- legal_scenario_1
- warehouse_scenario_1


In [31]:
from owlrl import DeductiveClosure, OWLRL_Semantics

In [32]:
print(len(g))

1142


In [33]:
asserted = Graph()
inferred = Graph()

asserted = asserted.parse("hi_ontology_linked.owl")
DeductiveClosure(OWLRL_Semantics).expand(g) 
inferred = g - asserted

In [39]:
print(f"Number of inferred sentences {len(inferred)}")
print(f"Probability of inferred sentences {len(inferred)/len(g)}")

Number of inferred sentences 2149
Probability of inferred sentences 0.6678060907395899
